# FMI 3.0 Export

Writing a model out as a Functional Mock-up Unit, for both interface types, and packaging a subsystem as a reusable component.

## What gets written

[FMI](https://fmi-standard.org/) is an open standard for moving simulation models between tools. `to_fmu` writes an **FMI 3.0 source FMU**: the model is lowered to C, wrapped in the FMI C layer, and zipped with a generated `modelDescription.xml` and `buildDescription.xml`.

*Source* FMU means the archive carries C rather than binaries, so the importing tool compiles it for its own platform.

One archive offers both interfaces:

- **Model Exchange** — the FMU hands out $\dot{x} = f(x, u, t)$ and the importer integrates it.
- **Co-Simulation** — the FMU integrates itself, with the solver baked in at export time, and the importer just advances it.

## The Model

A damped harmonic oscillator, which has a closed-form solution to check against:

$$\ddot{x} + 2\zeta\omega_0\dot{x} + \omega_0^2 x = 0, \qquad x(0) = 1,\ \dot{x}(0) = 0$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply the FastSim docs matplotlib style
plt.style.use('../fastsim_docs.mplstyle')

from fastsim import Simulation, Connection, Interface, Subsystem
from fastsim.blocks import ODE, Scope
from fastsim.solvers import RK4

## System Parameters

In [ ]:
omega_0 = 2 * np.pi      # natural frequency [rad/s]
zeta = 0.15              # damping ratio
T_END, DT = 3.0, 1e-3

omega_d = omega_0 * np.sqrt(1 - zeta**2)

def analytic(t):
    return np.exp(-zeta * omega_0 * t) * (
        np.cos(omega_d * t) + zeta * omega_0 / omega_d * np.sin(omega_d * t))

## Block Diagram

In [ ]:
def build():
    osc = ODE(lambda x, u, t: np.array([x[1], -2 * zeta * omega_0 * x[1] - omega_0**2 * x[0]]),
              initial_value=[1.0, 0.0])
    sco = Scope(labels=["x"], sampling_period=DT)
    sim = Simulation(
        blocks=[osc, sco],
        connections=[Connection(osc, sco)],
        Solver=RK4, dt=DT, log=False,
    )
    return sim, sco

## Simulation

The reference run, before exporting anything.

In [ ]:
sim, sco = build()
sim.run(T_END, reset=True, adaptive=False)
t_ref, [x_ref] = sco.read()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t_ref, x_ref)
ax.set_xlabel("time [s]")
ax.set_ylabel("x")
plt.show()

## Export

The keyword arguments become the default experiment in `modelDescription.xml` — what an importer uses when the user does not say otherwise.

In [ ]:
import tempfile, zipfile
from pathlib import Path

workdir = Path(tempfile.mkdtemp(prefix="fastsim_fmu_"))
fmu_path = workdir / "oscillator.fmu"

sim_export, _ = build()
sim_export.to_fmu(
    str(fmu_path),
    name="oscillator",
    start_time=0.0,
    stop_time=T_END,
    step_size=DT,
    tolerance=1e-8,
)

with zipfile.ZipFile(fmu_path) as z:
    for name in sorted(z.namelist()):
        print(name)

## What the Importer Sees

`modelDescription.xml` declares the interfaces on offer and the variables. Both `<ModelExchange>` and `<CoSimulation>` are there.

In [ ]:
with zipfile.ZipFile(fmu_path) as z:
    md_xml = z.read("modelDescription.xml").decode()

for line in md_xml.splitlines():
    if any(tag in line for tag in ("<ModelExchange", "<CoSimulation", "<Float64", "fmiVersion")):
        print(line.strip()[:110])

## Running It Back

[FMPy](https://github.com/CATIA-Systems/FMPy) is an independent FMI implementation, so it is a good way to check that the FMU really is one. It builds the C for this platform first, since a source FMU ships none.

In [ ]:
import shutil
import fmpy
from fmpy import read_model_description, simulate_fmu
from fmpy.build import build_platform_binary
from fmpy.validation import validate_fmu
from fastsim._fastsim import find_c_compiler

print("validate_fmu:", validate_fmu(str(fmu_path)) or "no problems")

md = read_model_description(str(fmu_path))
print(f"  fmiVersion    {md.fmiVersion}")
print(f"  ModelExchange {md.modelExchange is not None}")
print(f"  CoSimulation  {md.coSimulation is not None}")

unpacked = workdir / "unpacked"
with zipfile.ZipFile(fmu_path) as z:
    z.extractall(unpacked)

cc = find_c_compiler()
build_platform_binary(unpacked, cmake_options={"CMAKE_C_COMPILER": cc} if cc else {})

built = workdir / "oscillator_built.fmu"
shutil.make_archive(str(built.with_suffix("")), "zip", unpacked)
shutil.move(str(built.with_suffix(".zip")), str(built))

### Model Exchange

FMPy integrates the FMU with its own solver — CVode here, which FastSim never runs.

In [ ]:
res_me = simulate_fmu(str(built), start_time=0.0, stop_time=T_END, output_interval=DT * 10,
                      fmi_type="ModelExchange", solver="CVode", relative_tolerance=1e-10)
name = [n for n in res_me.dtype.names if n != "time"][0]
t_me, x_me = np.asarray(res_me["time"]), np.asarray(res_me[name])

### Co-Simulation

The same archive, the other way round: the FMU steps itself and FMPy only advances the clock.

In [ ]:
res_cs = simulate_fmu(str(built), start_time=0.0, stop_time=T_END, output_interval=DT * 10,
                      fmi_type="CoSimulation", step_size=DT)
t_cs, x_cs = np.asarray(res_cs["time"]), np.asarray(res_cs[name])

## Results

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_ref, x_ref, label="FastSim")
ax.plot(t_me, x_me, "--", label="FMU, Model Exchange")
ax.plot(t_cs, x_cs, ":", label="FMU, Co-Simulation")
ax.set_xlabel("time [s]")
ax.set_ylabel("x")
ax.legend()
plt.show()

## Verification

Against the closed-form solution, which involves neither the FMU nor FastSim.

In [ ]:
for label, t, x in (("FastSim", t_ref, x_ref),
                    ("FMU, Model Exchange", t_me, x_me),
                    ("FMU, Co-Simulation", t_cs, x_cs)):
    print(f"{label:<22} worst |x - analytic| = {np.max(np.abs(x - analytic(t))):.3e}")

## Exporting a Subsystem

A whole simulation is a closed model — no inputs, nothing to wire it to. A `Subsystem` is the other case: a component with ports, which becomes an FMU with real input and output variables that someone else can drop into their model.

The component here is a driven spring-mass-damper: force in, its state out.

$$m\ddot{x} + c\dot{x} + kx = F(t)$$

In [ ]:
m_mass, c_damp, k_spring = 1.0, 0.4, 25.0

iface = Interface()
plant = ODE(
    lambda x, u, t: np.array([x[1], (u[0] - c_damp * x[1] - k_spring * x[0]) / m_mass]),
    initial_value=[0.0, 0.0],
)

suspension = Subsystem(
    [iface, plant],
    [Connection(iface, plant),      # the subsystem's input drives the plant
     Connection(plant[0], iface)],  # the plant's position leaves as its output
)

sub_path = workdir / "suspension.fmu"
suspension.to_fmu(str(sub_path), name="suspension", start_time=0.0, stop_time=5.0, step_size=DT)

The exported component has an `input` variable — this FMU is not self-contained, it is meant to be connected. Its outputs are the plant's, position first.

In [ ]:
md_sub = read_model_description(str(sub_path))
for v in md_sub.modelVariables:
    if v.causality in ("input", "output"):
        print(f"  {v.causality:<8} {v.name}")

## Verification

Driven at its undamped natural frequency $\omega_n = \sqrt{k/m}$, the steady-state amplitude of such a component is known:

$$|X| = \frac{F_0}{c\,\omega_n}$$

In [ ]:
from fastsim.blocks import SinusoidalSource

omega_n = np.sqrt(k_spring / m_mass)
F0 = 1.0

src = SinusoidalSource(frequency=omega_n / (2 * np.pi), amplitude=F0)
sco_s = Scope(labels=["x"], sampling_period=1e-3)

sim_s = Simulation(
    blocks=[src, suspension, sco_s],
    connections=[Connection(src, suspension), Connection(suspension, sco_s)],
    Solver=RK4, dt=1e-4, log=False,
)
# The envelope decays with tau = 2m/c = 5 s, so let it settle for several of
# those before measuring the steady state.
sim_s.run(40.0, reset=True, adaptive=False)

t_s, [x_s] = sco_s.read()
settled = t_s > 35.0
amplitude = 0.5 * np.ptp(x_s[settled])

print(f"measured steady-state amplitude : {amplitude:.6f} m")
print(f"F0 / (c * omega_n)              : {F0 / (c_damp * omega_n):.6f} m")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t_s, x_s)
ax.set_xlabel("time [s]")
ax.set_ylabel("x [m]")
plt.show()

## Published FMUs

The two FMUs from this page are published in the repository under
[`fmus/`](https://github.com/pathsim/fastsim/tree/master/fmus), each with a
reference solution (`_ref.csv` / `_ref.opt` / `_in.csv`) so other tools can
import and compare. `scripts/export_reference_fmus.py` regenerates them, and
`scripts/check_exported_fmus.py` replays this page's comparison end to end.